# Kaggle: Jailbreak AutoIntent — **few-shot отдельно**, **full отдельно**

Самодостаточный ноутбук: код пишется в `/kaggle/working/kaggle_jailbreak_standalone/`.

**Логирование в `metrics.json` (поле `extra`):**
- `eval_counts`: TP/FP/FN/TN, `fnr_jailbreak`, `fpr_safe`, размеры классов на eval
- `scores_eval_summary`: агрегаты margin/score по `predict_with_metadata` (для будущего sweep порога)
- `decision_module_attrs`: дамп порога/конфига decision из `simple_attrs.json` (если не слишком большой)

**Вывод в output ноутбука:** после каждой оценки при флагах `--print-hypothesis-log` и `--print-metrics-json`:
- блок **`HYPOTHESIS_LOG`** — компактный JSON для отчёта
- блок **`METRICS_JSON`** — полная строка как в `metrics.json`

**Порядок:** зависимости → запись файлов → HF → `prepare_data` → **`§6a` few-shot (10,20,50)** → **`§6b` full** → просмотр метрик.

Можно запускать только §6a или только §6b, если данные уже подготовлены.


## 1. Корневая папка



In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IS_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()

PROJECT_ROOT = WORK / "kaggle_jailbreak_standalone"
TASK_ROOT = PROJECT_ROOT / "tasks" / "jailbreak_detection"

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TASK_ROOT:", TASK_ROOT)



## 2. Зависимости



In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "torch==2.5.1",
        "torchvision==0.20.1",
        "torchaudio==2.5.1",
        "transformers==4.49.0",
        "datasets==3.6.0",
        "sentence-transformers==3.4.1",
        "autointent==0.2.0",
        "scikit-learn>=1.5,<2",
        "pandas>=2.0",
        "numpy>=1.24,<3",
        "matplotlib>=3.7",
        "seaborn>=0.12",
        "optuna>=4,<5",
        "wandb>=0.19,<1",
        "python-dotenv>=1,<2",
    ],
    check=True,
)



## 3. Записать код (`run_autointent.py` с расширенным `extra`)



In [ ]:
import json
from pathlib import Path

_P = json.loads('{"tasks/__init__.py": "", "tasks/jailbreak_detection/__init__.py": "", "tasks/jailbreak_detection/src/__init__.py": "\\"\\"\\"\\nJailbreak Detection utilities.\\n\\nModules:\\n- metrics: jailbreak detection metrics (F1, recall, over-refusal rate)\\n\\"\\"\\"\\n\\nfrom .metrics import (\\n    evaluate_jailbreak,\\n    over_refusal_rate,\\n    precision_oos,\\n    oos_recall,\\n    f1_oos,\\n)\\n\\n__all__ = [\\n    \\"evaluate_jailbreak\\",\\n    \\"over_refusal_rate\\",\\n    \\"precision_oos\\",\\n    \\"oos_recall\\",\\n    \\"f1_oos\\",\\n]\\n", "tasks/jailbreak_detection/src/metrics.py": "\\"\\"\\"\\nMetrics for Jailbreak Detection evaluation.\\n\\nPrimary metrics:\\n  - f1              : F1-score on jailbreak class\\n  - precision       : TP / (TP + FP) on jailbreak class\\n  - recall          : TPR on jailbreak class\\n  - over_refusal_rate : FPR on safe prompts (false positive rate)\\n\\nSubset metrics (when data_types available):\\n  - recall_vanilla_harmful     : recall on vanilla_harmful subset\\n  - recall_adversarial_harmful : recall on adversarial_harmful subset\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport numpy as np\\nfrom sklearn.metrics import f1_score, recall_score\\n\\n\\ndef oos_recall(\\n    y_true: np.ndarray,\\n    y_pred: np.ndarray,\\n    oos_label: int = 1,\\n) -> float:\\n    \\"\\"\\"Recall on jailbreak/OOS class (TPR).\\"\\"\\"\\n    y_true_bin = (np.asarray(y_true) == oos_label).astype(int)\\n    y_pred_bin = (np.asarray(y_pred) == oos_label).astype(int)\\n    return float(recall_score(y_true_bin, y_pred_bin, zero_division=0))\\n\\n\\ndef f1_oos(\\n    y_true: np.ndarray,\\n    y_pred: np.ndarray,\\n    oos_label: int = 1,\\n) -> float:\\n    \\"\\"\\"F1-score on jailbreak/OOS class.\\"\\"\\"\\n    y_true_bin = (np.asarray(y_true) == oos_label).astype(int)\\n    y_pred_bin = (np.asarray(y_pred) == oos_label).astype(int)\\n    return float(f1_score(y_true_bin, y_pred_bin, zero_division=0))\\n\\n\\ndef over_refusal_rate(\\n    y_true: np.ndarray,\\n    y_pred: np.ndarray,\\n    oos_label: int = 1,\\n) -> float:\\n    \\"\\"\\"\\n    Over-refusal Rate: fraction of safe prompts incorrectly blocked.\\n\\n    FPR on safe (benign) prompts:\\n      = FP / (FP + TN)\\n    where positive = jailbreak,\\n    computed only on examples with y_true != oos_label.\\n\\n    Used in:\\n    - WildGuard (NeurIPS 2024) as \\"Refusal Rate on benign prompts\\"\\n    - XSTest (NAACL 2024) as \\"Over-refusal Rate\\"\\n\\n    Args:\\n        y_true: true labels\\n        y_pred: predicted labels\\n        oos_label: jailbreak class label (default 1)\\n\\n    Returns:\\n        over-refusal rate, float in [0, 1]\\n    \\"\\"\\"\\n    y_true = np.asarray(y_true)\\n    y_pred = np.asarray(y_pred)\\n\\n    # Only safe examples (y_true != oos_label)\\n    safe_mask = y_true != oos_label\\n    n_safe = safe_mask.sum()\\n\\n    if n_safe == 0:\\n        return 0.0\\n\\n    # FP: safe examples predicted as jailbreak\\n    false_positives = ((y_pred == oos_label) & safe_mask).sum()\\n\\n    return float(false_positives / n_safe)\\n\\n\\ndef precision_oos(\\n    y_true: np.ndarray,\\n    y_pred: np.ndarray,\\n    oos_label: int = 1,\\n) -> float:\\n    \\"\\"\\"\\n    Precision on jailbreak class:\\n      TP / (TP + FP)\\n    Fraction of blocked prompts that are actually jailbreak.\\n\\n    Args:\\n        y_true: true labels\\n        y_pred: predicted labels\\n        oos_label: jailbreak class label (default 1)\\n\\n    Returns:\\n        precision, float in [0, 1]\\n    \\"\\"\\"\\n    y_true = np.asarray(y_true)\\n    y_pred = np.asarray(y_pred)\\n\\n    # Predicted as jailbreak\\n    pred_oos_mask = y_pred == oos_label\\n    n_pred_oos = pred_oos_mask.sum()\\n\\n    if n_pred_oos == 0:\\n        return 0.0\\n\\n    # TP: correctly predicted jailbreak\\n    true_positives = ((y_true == oos_label) & pred_oos_mask).sum()\\n\\n    return float(true_positives / n_pred_oos)\\n\\n\\ndef evaluate_jailbreak(\\n    y_true: np.ndarray,\\n    y_pred: np.ndarray,\\n    data_types: np.ndarray | None = None,\\n    oos_label: int = 1,\\n) -> dict:\\n    \\"\\"\\"\\n    Full metric suite for Jailbreak Detection.\\n\\n    Args:\\n        y_true: true labels (1 = jailbreak, 0 = safe)\\n        y_pred: predicted labels\\n        data_types: array of data_type strings from WildJailbreak.\\n                    If None, subset metrics are not computed.\\n        oos_label: jailbreak class label (default 1)\\n\\n    Returns dict:\\n        \\"f1\\"                         : f1_oos(...)\\n        \\"precision\\"                  : precision_oos(...)\\n        \\"recall\\"                     : oos_recall(...)\\n        \\"over_refusal_rate\\"          : over_refusal_rate(...)\\n\\n        If data_types provided, additionally:\\n        \\"recall_vanilla_harmful\\"     : recall on vanilla_harmful subset\\n        \\"recall_adversarial_harmful\\" : recall on adversarial_harmful subset\\n    \\"\\"\\"\\n    y_true = np.asarray(y_true)\\n    y_pred = np.asarray(y_pred)\\n\\n    result = {\\n        \\"f1\\": f1_oos(y_true, y_pred, oos_label),\\n        \\"precision\\": precision_oos(y_true, y_pred, oos_label),\\n        \\"recall\\": oos_recall(y_true, y_pred, oos_label),\\n        \\"over_refusal_rate\\": over_refusal_rate(y_true, y_pred, oos_label),\\n    }\\n\\n    if data_types is not None:\\n        data_types = np.asarray(data_types)\\n\\n        # Recall on vanilla_harmful subset\\n        vanilla_harmful_mask = data_types == \\"vanilla_harmful\\"\\n        if vanilla_harmful_mask.sum() > 0:\\n            result[\\"recall_vanilla_harmful\\"] = oos_recall(\\n                y_true[vanilla_harmful_mask],\\n                y_pred[vanilla_harmful_mask],\\n                oos_label,\\n            )\\n\\n        # Recall on adversarial_harmful subset\\n        adversarial_harmful_mask = data_types == \\"adversarial_harmful\\"\\n        if adversarial_harmful_mask.sum() > 0:\\n            result[\\"recall_adversarial_harmful\\"] = oos_recall(\\n                y_true[adversarial_harmful_mask],\\n                y_pred[adversarial_harmful_mask],\\n                oos_label,\\n            )\\n\\n    return result\\n\\n\\nif __name__ == \\"__main__\\":\\n    # Smoke tests\\n    y_true = np.array([0, 0, 0, 0, 1, 1])  # 4 safe, 2 jailbreak\\n    y_pred = np.array([0, 1, 0, 1, 1, 1])  # 2 safe incorrectly blocked\\n\\n    orr = over_refusal_rate(y_true, y_pred, oos_label=1)\\n    assert orr == 0.5, f\\"Expected 0.5, got {orr}\\"\\n    print(f\\"over_refusal_rate test passed: {orr}\\")\\n\\n    prec = precision_oos(y_true, y_pred, oos_label=1)\\n    assert prec == 0.5, f\\"Expected 0.5, got {prec}\\"\\n    print(f\\"precision_oos test passed: {prec}\\")\\n\\n    metrics = evaluate_jailbreak(y_true, y_pred, oos_label=1)\\n    print(f\\"evaluate_jailbreak test passed: {metrics}\\")\\n", "tasks/jailbreak_detection/scripts/prepare_data.py": "\\"\\"\\"\\nПодготовка данных WildJailbreak для экспериментов Jailbreak Detection.\\n\\nЗагружает данные из HuggingFace (allenai/wildjailbreak),\\nконвертирует в бинарный формат (safe/jailbreak),\\nсоздаёт few-shot сплиты и full-train подвыборки в формате AutoIntent.\\n\\nИспользование:\\n    # Few-shot сплиты (по умолчанию)\\n    python scripts/prepare_data.py\\n\\n    # Full-train 100K подвыборки для AutoML\\n    python scripts/prepare_data.py --full_subset\\n\\n    Датасет allenai/wildjailbreak на Hugging Face закрытый (gated): на странице датасета\\n    нужно принять условия, затем авторизация — huggingface-cli login или переменная HF_TOKEN.\\n\\n    # С кастомными параметрами\\n    python scripts/prepare_data.py \\\\\\n        --raw_train data/raw/wildjailbreak_train.jsonl \\\\\\n        --raw_eval data/raw/wildjailbreak_eval.jsonl \\\\\\n        --output_dir data/processed \\\\\\n        --n_shots 10 20 50 \\\\\\n        --seeds 42 123 456\\n\\nВыходные файлы:\\n    data/raw/wildjailbreak_train.jsonl   — сырые train данные\\n    data/raw/wildjailbreak_eval.jsonl    — сырые eval данные\\n    data/processed/wildjailbreak_eval_binary.jsonl — eval с binary_label\\n    data/processed/train_shot{N}_seed{S}.json — few-shot сплиты (AutoIntent)\\n    data/processed/test.json             — тестовый сплит (AutoIntent)\\n    data/processed/wildjailbreak_full100k_seed{S}.json — full-train 100K (AutoIntent)\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport os\\nimport warnings\\nfrom pathlib import Path\\n\\nimport numpy as np\\nfrom datasets import load_dataset\\nfrom datasets.exceptions import DatasetNotFoundError\\n\\n\\n# Constants\\nHARMFUL_TYPES = {\\"vanilla_harmful\\", \\"adversarial_harmful\\"}\\nBENIGN_TYPES = {\\"vanilla_benign\\", \\"adversarial_benign\\"}\\nVANILLA_TYPES = {\\"vanilla_harmful\\", \\"vanilla_benign\\"}\\nADVERSARIAL_TYPES = {\\"adversarial_harmful\\", \\"adversarial_benign\\"}\\nDEFAULT_N_SHOTS = [10, 20, 50]\\nDEFAULT_SEEDS = [42, 123, 456]\\n\\n\\ndef _hf_token() -> str | None:\\n    \\"\\"\\"Токен для gated-датасетов (HF_TOKEN или HUGGING_FACE_HUB_TOKEN).\\"\\"\\"\\n    return os.environ.get(\\"HF_TOKEN\\") or os.environ.get(\\"HUGGING_FACE_HUB_TOKEN\\")\\n\\n\\ndef _print_hf_gated_help() -> None:\\n    print(\\"\\\\n\\" + \\"=\\" * 60)\\n    print(\\"WildJailbreak (allenai/wildjailbreak) — gated dataset на Hugging Face.\\")\\n    print(\\"1) Откройте страницу датасета и примите условия доступа.\\")\\n    print(\\"2) Токен: https://huggingface.co/settings/tokens\\")\\n    print(\\"3) Выполните:  huggingface-cli login\\")\\n    print(\\"   или:        export HF_TOKEN=<ваш_токен>\\")\\n    print(\\"=\\" * 60 + \\"\\\\n\\")\\n\\n\\ndef get_binary_label(data_type: str) -> str:\\n    \\"\\"\\"Convert data_type to binary label.\\"\\"\\"\\n    if data_type in HARMFUL_TYPES:\\n        return \\"jailbreak\\"\\n    elif data_type in BENIGN_TYPES:\\n        return \\"safe\\"\\n    else:\\n        raise ValueError(f\\"Unknown data_type: {data_type}\\")\\n\\n\\ndef get_prompt_train(example: dict) -> str:\\n    \\"\\"\\"\\n    Get prompt text from train example.\\n    Train has columns: vanilla, adversarial, completion, data_type.\\n    If adversarial is non-empty, use it; otherwise use vanilla.\\n    \\"\\"\\"\\n    adversarial = example.get(\\"adversarial\\", \\"\\")\\n    if adversarial and adversarial.strip():\\n        return adversarial\\n    return example[\\"vanilla\\"]\\n\\n\\ndef get_prompt_eval(example: dict) -> str:\\n    \\"\\"\\"\\n    Get prompt text from eval example.\\n    Eval has columns: adversarial, label, data_type.\\n    The \'adversarial\' column contains the prompt.\\n    \\"\\"\\"\\n    return example[\\"adversarial\\"]\\n\\n\\ndef load_and_save_raw(output_dir: Path) -> tuple[list[dict], list[dict]]:\\n    \\"\\"\\"\\n    Load WildJailbreak from HuggingFace and save raw data.\\n\\n    Returns:\\n        (train_data, eval_data) as lists of dicts\\n    \\"\\"\\"\\n    raw_dir = output_dir.parent / \\"raw\\"\\n    raw_dir.mkdir(parents=True, exist_ok=True)\\n\\n    print(\\"Loading WildJailbreak from HuggingFace...\\")\\n\\n    dl_mode = os.environ.get(\\"HF_DATASETS_DOWNLOAD_MODE\\")  # напр. force_redownload\\n    _ld_extra: dict = {\\"token\\": _hf_token()}\\n    if dl_mode:\\n        _ld_extra[\\"download_mode\\"] = dl_mode\\n\\n    # Load train split (has columns: vanilla, adversarial, completion, data_type)\\n    print(\\"Loading train split...\\")\\n    try:\\n        train_dataset = load_dataset(\\n            \\"allenai/wildjailbreak\\",\\n            data_files={\\"train\\": \\"train/train.tsv\\"},\\n            split=\\"train\\",\\n            delimiter=\\"\\\\t\\",\\n            keep_default_na=False,\\n            **_ld_extra,\\n        )\\n    except DatasetNotFoundError:\\n        _print_hf_gated_help()\\n        raise\\n\\n    train_data = list(train_dataset)\\n    print(f\\"Train loaded: {len(train_data)} examples\\")\\n\\n    # Load eval split separately (has columns: adversarial, label, data_type)\\n    # Note: eval split has different schema - adversarial contains the prompt\\n    print(\\"Loading eval split...\\")\\n    try:\\n        eval_dataset = load_dataset(\\n            \\"allenai/wildjailbreak\\",\\n            data_files={\\"test\\": \\"eval/eval.tsv\\"},\\n            split=\\"test\\",\\n            delimiter=\\"\\\\t\\",\\n            keep_default_na=False,\\n            **_ld_extra,\\n        )\\n    except DatasetNotFoundError:\\n        _print_hf_gated_help()\\n        raise\\n    eval_data = list(eval_dataset)\\n    print(f\\"Eval loaded: {len(eval_data)} examples\\")\\n\\n    # Save raw data\\n    train_path = raw_dir / \\"wildjailbreak_train.jsonl\\"\\n    eval_path = raw_dir / \\"wildjailbreak_eval.jsonl\\"\\n\\n    with open(train_path, \\"w\\", encoding=\\"utf-8\\") as f:\\n        for item in train_data:\\n            f.write(json.dumps(item, ensure_ascii=False) + \\"\\\\n\\")\\n\\n    with open(eval_path, \\"w\\", encoding=\\"utf-8\\") as f:\\n        for item in eval_data:\\n            f.write(json.dumps(item, ensure_ascii=False) + \\"\\\\n\\")\\n\\n    print(f\\"Saved raw train: {train_path} ({len(train_data)} examples)\\")\\n    print(f\\"Saved raw eval:  {eval_path} ({len(eval_data)} examples)\\")\\n\\n    return train_data, eval_data\\n\\n\\ndef load_raw_from_files(raw_train: Path, raw_eval: Path) -> tuple[list[dict], list[dict]]:\\n    \\"\\"\\"Load raw data from existing JSONL files.\\"\\"\\"\\n    train_data = []\\n    with open(raw_train, \\"r\\", encoding=\\"utf-8\\") as f:\\n        for line in f:\\n            train_data.append(json.loads(line))\\n\\n    eval_data = []\\n    with open(raw_eval, \\"r\\", encoding=\\"utf-8\\") as f:\\n        for line in f:\\n            eval_data.append(json.loads(line))\\n\\n    return train_data, eval_data\\n\\n\\ndef process_eval_binary(eval_data: list[dict], output_dir: Path) -> dict:\\n    \\"\\"\\"\\n    Process eval data: add binary_label and prompt columns.\\n    Save to wildjailbreak_eval_binary.jsonl.\\n\\n    Returns:\\n        Distribution statistics\\n    \\"\\"\\"\\n    processed = []\\n    stats = {\\"data_type\\": {}, \\"binary_label\\": {\\"safe\\": 0, \\"jailbreak\\": 0}}\\n\\n    for example in eval_data:\\n        data_type = example[\\"data_type\\"]\\n        binary_label = get_binary_label(data_type)\\n        prompt = get_prompt_eval(example)\\n\\n        processed.append({\\n            **example,\\n            \\"binary_label\\": binary_label,\\n            \\"prompt\\": prompt\\n        })\\n\\n        # Update stats\\n        stats[\\"data_type\\"][data_type] = stats[\\"data_type\\"].get(data_type, 0) + 1\\n        stats[\\"binary_label\\"][binary_label] += 1\\n\\n    # Save\\n    output_path = output_dir / \\"wildjailbreak_eval_binary.jsonl\\"\\n    with open(output_path, \\"w\\", encoding=\\"utf-8\\") as f:\\n        for item in processed:\\n            f.write(json.dumps(item, ensure_ascii=False) + \\"\\\\n\\")\\n\\n    print(f\\"\\\\nSaved eval with binary labels: {output_path}\\")\\n\\n    return stats\\n\\n\\ndef create_fewshot_split(\\n    train_data: list[dict],\\n    n_shots: int,\\n    seed: int,\\n    output_dir: Path\\n) -> None:\\n    \\"\\"\\"\\n    Create stratified few-shot split in AutoIntent format.\\n\\n    AutoIntent format:\\n    {\\n        \\"intents\\": [{\\"id\\": 0, \\"name\\": \\"safe\\", \\"utterances\\": [...]}],\\n        \\"oos_utterances\\": [...]\\n    }\\n    \\"\\"\\"\\n    np.random.seed(seed)\\n\\n    # Group by binary label\\n    safe_examples = []\\n    jailbreak_examples = []\\n\\n    for example in train_data:\\n        prompt = get_prompt_train(example)\\n        binary_label = get_binary_label(example[\\"data_type\\"])\\n\\n        if binary_label == \\"safe\\":\\n            safe_examples.append(prompt)\\n        else:\\n            jailbreak_examples.append(prompt)\\n\\n    # Sample n_shots per class\\n    safe_sampled = list(np.random.choice(\\n        safe_examples,\\n        size=min(n_shots, len(safe_examples)),\\n        replace=False\\n    ))\\n    jailbreak_sampled = list(np.random.choice(\\n        jailbreak_examples,\\n        size=min(n_shots, len(jailbreak_examples)),\\n        replace=False\\n    ))\\n\\n    # Create AutoIntent format\\n    autointent_data = {\\n        \\"intents\\": [\\n            {\\n                \\"id\\": 0,\\n                \\"name\\": \\"safe\\",\\n                \\"utterances\\": safe_sampled\\n            }\\n        ],\\n        \\"oos_utterances\\": jailbreak_sampled\\n    }\\n\\n    # Save\\n    output_path = output_dir / f\\"train_shot{n_shots}_seed{seed}.json\\"\\n    with open(output_path, \\"w\\", encoding=\\"utf-8\\") as f:\\n        json.dump(autointent_data, f, ensure_ascii=False, indent=2)\\n\\n\\ndef create_test_split(eval_data: list[dict], output_dir: Path) -> None:\\n    \\"\\"\\"\\n    Create test split in AutoIntent format.\\n\\n    Format:\\n    {\\n        \\"utterances\\": [...],\\n        \\"labels\\": [...]  # 0 = safe, 1 = jailbreak/OOS\\n    }\\n    \\"\\"\\"\\n    utterances = []\\n    labels = []\\n\\n    for example in eval_data:\\n        prompt = get_prompt_eval(example)\\n        binary_label = get_binary_label(example[\\"data_type\\"])\\n\\n        utterances.append(prompt)\\n        labels.append(0 if binary_label == \\"safe\\" else 1)\\n\\n    test_data = {\\n        \\"utterances\\": utterances,\\n        \\"labels\\": labels\\n    }\\n\\n    # Save\\n    output_path = output_dir / \\"test.json\\"\\n    with open(output_path, \\"w\\", encoding=\\"utf-8\\") as f:\\n        json.dump(test_data, f, ensure_ascii=False, indent=2)\\n\\n    print(f\\"Saved test split: {output_path}\\")\\n\\n\\ndef print_distribution(stats: dict) -> None:\\n    \\"\\"\\"Print class distribution.\\"\\"\\"\\n    print(\\"\\\\n=== Eval Distribution ===\\")\\n    print(\\"\\\\nBy data_type:\\")\\n    for dtype, count in sorted(stats[\\"data_type\\"].items()):\\n        print(f\\"  {dtype}: {count}\\")\\n\\n    print(\\"\\\\nBy binary_label:\\")\\n    for label, count in sorted(stats[\\"binary_label\\"].items()):\\n        print(f\\"  {label}: {count}\\")\\n\\n    total = sum(stats[\\"binary_label\\"].values())\\n    print(f\\"\\\\nTotal: {total}\\")\\n\\n\\ndef make_full_train_subset(\\n    train_data: list[dict],\\n    n_total: int = 100_000,\\n    seed: int = 42\\n) -> list[dict]:\\n    \\"\\"\\"\\n    Create stratified subset of train data for full-train AutoML experiments.\\n\\n    Stratification:\\n    - 50/50 balance between safe and jailbreak classes\\n    - Within each class, preserve original vanilla/adversarial ratio\\n\\n    Reference: Shi et al. 2021 (NeurIPS D&B) downsampled large text datasets\\n    (jigsaw toxicity, mercari) to 100K for AutoML computational tractability.\\n\\n    Args:\\n        train_data: Full train dataset as list of dicts\\n        n_total: Target total size (default 100K)\\n        seed: Random seed for reproducibility\\n\\n    Returns:\\n        Stratified subset of train_data\\n    \\"\\"\\"\\n    rng = np.random.default_rng(seed)\\n\\n    # Group examples by stratum (data_type)\\n    strata = {\\n        \\"vanilla_harmful\\": [],\\n        \\"adversarial_harmful\\": [],\\n        \\"vanilla_benign\\": [],\\n        \\"adversarial_benign\\": [],\\n    }\\n\\n    for idx, example in enumerate(train_data):\\n        data_type = example[\\"data_type\\"]\\n        strata[data_type].append(idx)\\n\\n    # Target: 50K per class\\n    n_per_class = n_total // 2\\n\\n    # Calculate proportions within each class\\n    # Jailbreak class (harmful)\\n    n_vanilla_harmful = len(strata[\\"vanilla_harmful\\"])\\n    n_adversarial_harmful = len(strata[\\"adversarial_harmful\\"])\\n    n_harmful_total = n_vanilla_harmful + n_adversarial_harmful\\n\\n    prop_vanilla_harmful = n_vanilla_harmful / n_harmful_total\\n    prop_adversarial_harmful = n_adversarial_harmful / n_harmful_total\\n\\n    # Safe class (benign)\\n    n_vanilla_benign = len(strata[\\"vanilla_benign\\"])\\n    n_adversarial_benign = len(strata[\\"adversarial_benign\\"])\\n    n_benign_total = n_vanilla_benign + n_adversarial_benign\\n\\n    prop_vanilla_benign = n_vanilla_benign / n_benign_total\\n    prop_adversarial_benign = n_adversarial_benign / n_benign_total\\n\\n    # Calculate target counts per stratum\\n    targets = {\\n        \\"vanilla_harmful\\": int(round(n_per_class * prop_vanilla_harmful)),\\n        \\"adversarial_harmful\\": int(round(n_per_class * prop_adversarial_harmful)),\\n        \\"vanilla_benign\\": int(round(n_per_class * prop_vanilla_benign)),\\n        \\"adversarial_benign\\": int(round(n_per_class * prop_adversarial_benign)),\\n    }\\n\\n    # Adjust for rounding to hit exact n_per_class per binary class\\n    harmful_sum = targets[\\"vanilla_harmful\\"] + targets[\\"adversarial_harmful\\"]\\n    if harmful_sum != n_per_class:\\n        targets[\\"adversarial_harmful\\"] += (n_per_class - harmful_sum)\\n\\n    benign_sum = targets[\\"vanilla_benign\\"] + targets[\\"adversarial_benign\\"]\\n    if benign_sum != n_per_class:\\n        targets[\\"adversarial_benign\\"] += (n_per_class - benign_sum)\\n\\n    # Sample from each stratum\\n    sampled_indices = []\\n    for stratum_name, target_n in targets.items():\\n        available = strata[stratum_name]\\n        actual_n = min(target_n, len(available))\\n\\n        if actual_n < target_n:\\n            warnings.warn(\\n                f\\"Stratum \'{stratum_name}\' has only {len(available)} examples, \\"\\n                f\\"requested {target_n}. Using all available.\\"\\n            )\\n\\n        chosen = rng.choice(available, size=actual_n, replace=False)\\n        sampled_indices.extend(chosen)\\n\\n    # Shuffle the combined sample\\n    rng.shuffle(sampled_indices)\\n\\n    # Extract sampled examples\\n    sampled_data = [train_data[i] for i in sampled_indices]\\n\\n    return sampled_data\\n\\n\\ndef create_full_train_split(\\n    train_data: list[dict],\\n    n_total: int,\\n    seed: int,\\n    output_dir: Path\\n) -> dict:\\n    \\"\\"\\"\\n    Create stratified full-train subset in AutoIntent format.\\n\\n    Args:\\n        train_data: Full train dataset\\n        n_total: Target total size\\n        seed: Random seed\\n        output_dir: Output directory\\n\\n    Returns:\\n        Statistics dict for verification\\n    \\"\\"\\"\\n    # Get stratified subset\\n    subset = make_full_train_subset(train_data, n_total=n_total, seed=seed)\\n\\n    # Convert to AutoIntent format\\n    safe_utterances = []\\n    jailbreak_utterances = []\\n    stats = {\\n        \\"total\\": len(subset),\\n        \\"safe\\": 0,\\n        \\"jailbreak\\": 0,\\n        \\"vanilla_harmful\\": 0,\\n        \\"adversarial_harmful\\": 0,\\n        \\"vanilla_benign\\": 0,\\n        \\"adversarial_benign\\": 0,\\n    }\\n\\n    for example in subset:\\n        prompt = get_prompt_train(example)\\n        data_type = example[\\"data_type\\"]\\n        binary_label = get_binary_label(data_type)\\n\\n        stats[data_type] += 1\\n\\n        if binary_label == \\"safe\\":\\n            safe_utterances.append(prompt)\\n            stats[\\"safe\\"] += 1\\n        else:\\n            jailbreak_utterances.append(prompt)\\n            stats[\\"jailbreak\\"] += 1\\n\\n    # Create AutoIntent format (same as few-shot)\\n    autointent_data = {\\n        \\"intents\\": [\\n            {\\n                \\"id\\": 0,\\n                \\"name\\": \\"safe\\",\\n                \\"utterances\\": safe_utterances\\n            }\\n        ],\\n        \\"oos_utterances\\": jailbreak_utterances\\n    }\\n\\n    # Save\\n    output_path = output_dir / f\\"wildjailbreak_full100k_seed{seed}.json\\"\\n    with open(output_path, \\"w\\", encoding=\\"utf-8\\") as f:\\n        json.dump(autointent_data, f, ensure_ascii=False, indent=2)\\n\\n    return stats\\n\\n\\ndef verify_full_train_subsets(output_dir: Path, seeds: list[int]) -> None:\\n    \\"\\"\\"\\n    Verify full-train subsets: size, balance, and cross-seed overlap.\\n\\n    Args:\\n        output_dir: Directory containing the subset files\\n        seeds: List of seeds used\\n    \\"\\"\\"\\n    print(\\"\\\\n\\" + \\"=\\" * 60)\\n    print(\\"VERIFICATION: Full-Train 100K Subsets\\")\\n    print(\\"=\\" * 60)\\n\\n    all_indices = {}\\n\\n    for seed in seeds:\\n        path = output_dir / f\\"wildjailbreak_full100k_seed{seed}.json\\"\\n        with open(path, \\"r\\", encoding=\\"utf-8\\") as f:\\n            data = json.load(f)\\n\\n        safe = data[\\"intents\\"][0][\\"utterances\\"]\\n        jailbreak = data[\\"oos_utterances\\"]\\n        total = len(safe) + len(jailbreak)\\n\\n        # Store utterances for overlap calculation\\n        all_indices[seed] = set(safe + jailbreak)\\n\\n        print(f\\"\\\\n--- Seed {seed} ---\\")\\n        print(f\\"Total: {total:,}\\")\\n        print(f\\"Safe: {len(safe):,} ({100*len(safe)/total:.1f}%)\\")\\n        print(f\\"Jailbreak: {len(jailbreak):,} ({100*len(jailbreak)/total:.1f}%)\\")\\n\\n    # Calculate pairwise overlap\\n    print(\\"\\\\n--- Cross-seed Overlap ---\\")\\n    print(\\"(Expected ~38K based on 100K×100K/261K)\\")\\n    seed_list = list(seeds)\\n    for i in range(len(seed_list)):\\n        for j in range(i + 1, len(seed_list)):\\n            s1, s2 = seed_list[i], seed_list[j]\\n            overlap = len(all_indices[s1] & all_indices[s2])\\n            print(f\\"Seed {s1} ∩ Seed {s2}: {overlap:,} utterances\\")\\n\\n    print(\\"=\\" * 60)\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser(\\n        description=\\"Prepare WildJailbreak data for jailbreak detection experiments\\"\\n    )\\n    parser.add_argument(\\n        \\"--raw_train\\",\\n        type=Path,\\n        help=\\"Path to existing raw train JSONL (skip download if provided)\\"\\n    )\\n    parser.add_argument(\\n        \\"--raw_eval\\",\\n        type=Path,\\n        help=\\"Path to existing raw eval JSONL (skip download if provided)\\"\\n    )\\n    parser.add_argument(\\n        \\"--output_dir\\",\\n        type=Path,\\n        default=None,\\n        help=\\"Output directory for processed data\\"\\n    )\\n    parser.add_argument(\\n        \\"--n_shots\\",\\n        type=int,\\n        nargs=\\"+\\",\\n        default=DEFAULT_N_SHOTS,\\n        help=\\"Number of shots per class for few-shot splits\\"\\n    )\\n    parser.add_argument(\\n        \\"--seeds\\",\\n        type=int,\\n        nargs=\\"+\\",\\n        default=DEFAULT_SEEDS,\\n        help=\\"Random seeds for sampling\\"\\n    )\\n    parser.add_argument(\\n        \\"--full_subset\\",\\n        action=\\"store_true\\",\\n        help=\\"Create 100K stratified full-train subsets for AutoML experiments\\"\\n    )\\n    parser.add_argument(\\n        \\"--full_subset_size\\",\\n        type=int,\\n        default=100_000,\\n        help=\\"Size of full-train subset (default: 100000)\\"\\n    )\\n\\n    args = parser.parse_args()\\n\\n    # Determine output directory\\n    if args.output_dir:\\n        output_dir = args.output_dir\\n    else:\\n        script_dir = Path(__file__).parent\\n        output_dir = script_dir.parent / \\"data\\" / \\"processed\\"\\n\\n    output_dir.mkdir(parents=True, exist_ok=True)\\n\\n    # Determine raw data paths\\n    script_dir = Path(__file__).parent\\n    default_raw_dir = script_dir.parent / \\"data\\" / \\"raw\\"\\n    raw_train = args.raw_train or default_raw_dir / \\"wildjailbreak_train.jsonl\\"\\n    raw_eval = args.raw_eval or default_raw_dir / \\"wildjailbreak_eval.jsonl\\"\\n\\n    # Load data\\n    if raw_train.exists() and raw_eval.exists():\\n        print(f\\"Loading from existing files...\\")\\n        train_data, eval_data = load_raw_from_files(raw_train, raw_eval)\\n        print(f\\"Loaded train: {len(train_data)} examples\\")\\n        print(f\\"Loaded eval:  {len(eval_data)} examples\\")\\n    else:\\n        train_data, eval_data = load_and_save_raw(output_dir)\\n\\n    if args.full_subset:\\n        # Create full-train 100K subsets only\\n        print(\\"\\\\n=== Creating Full-Train 100K Subsets ===\\")\\n        print(f\\"Target size: {args.full_subset_size:,}\\")\\n        print(f\\"Seeds: {args.seeds}\\")\\n\\n        for seed in args.seeds:\\n            stats = create_full_train_split(\\n                train_data,\\n                n_total=args.full_subset_size,\\n                seed=seed,\\n                output_dir=output_dir\\n            )\\n            print(f\\"\\\\nCreated wildjailbreak_full100k_seed{seed}.json\\")\\n            print(f\\"  Total: {stats[\'total\']:,}\\")\\n            print(f\\"  Safe: {stats[\'safe\']:,} \\"\\n                  f\\"(vanilla: {stats[\'vanilla_benign\']:,}, \\"\\n                  f\\"adversarial: {stats[\'adversarial_benign\']:,})\\")\\n            print(f\\"  Jailbreak: {stats[\'jailbreak\']:,} \\"\\n                  f\\"(vanilla: {stats[\'vanilla_harmful\']:,}, \\"\\n                  f\\"adversarial: {stats[\'adversarial_harmful\']:,})\\")\\n\\n        # Verification\\n        verify_full_train_subsets(output_dir, args.seeds)\\n\\n    else:\\n        # Default: create few-shot splits and test split\\n        # Process eval with binary labels\\n        stats = process_eval_binary(eval_data, output_dir)\\n        print_distribution(stats)\\n\\n        # Create few-shot splits\\n        print(\\"\\\\n=== Creating Few-Shot Splits ===\\")\\n        for n_shots in args.n_shots:\\n            for seed in args.seeds:\\n                create_fewshot_split(train_data, n_shots, seed, output_dir)\\n                print(f\\"Created train_shot{n_shots}_seed{seed}.json\\")\\n\\n        # Create test split\\n        print(\\"\\\\n=== Creating Test Split ===\\")\\n        create_test_split(eval_data, output_dir)\\n\\n    print(f\\"\\\\nDone! All files saved to {output_dir}/\\")\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "tasks/jailbreak_detection/scripts/run_autointent.py": "\\"\\"\\"\\nЗапуск AutoIntent на WildJailbreak (обучение + оценка).\\n\\nИспользование:\\n    # Pilot (маленький embedder, быстрая валидация)\\n    python scripts/run_autointent.py --mode fewshot --n_shots 10 --seed 42 --pilot\\n\\n    # Final (оригинальный embedder e5)\\n    python scripts/run_autointent.py --mode fewshot --n_shots 10 --seed 42\\n\\n    # AutoML без фиксации embedder (медленнее, потенциально лучше)\\n    python scripts/run_autointent.py --mode fewshot --n_shots 10 --seed 42 --no-fix-embedder\\n\\n    # Только оценка (модель должна существовать)\\n    python scripts/run_autointent.py --mode fewshot --n_shots 10 --seed 42 --eval-only\\n\\n    # Full train (100K stratified wildjailbreak_full100k_seed{S}.json из prepare_data --full_subset)\\n    python scripts/run_autointent.py --mode full --seed 42\\n\\n    # Full на всех сидах (42, 123, 456) подряд\\n    python scripts/run_autointent.py --mode full --all-seeds\\n\\nДанные:\\n    - Few-shot train: data/processed/train_shot{N}_seed{S}.json\\n    - Full train: data/processed/wildjailbreak_full100k_seed{S}.json\\n      (это 100K стратифицированная подвыборка, не весь сырой WildJailbreak train;\\n      готовится скриптом prepare_data.py --full_subset из‑за размера и времени AutoML)\\n    - Test: data/processed/test.json (для inference)\\n    - Eval binary: data/processed/wildjailbreak_eval_binary.jsonl (для метрик)\\n\\nРезультаты:\\n    - Модель few-shot (фикс. e5): runs/autointent_classic-light_{N}shot_seed{S}/\\n    - Модель few-shot (автоэмбеддер): runs/autointent_classic-light_autoembedder_{N}shot_seed{S}/\\n    - Модель full: runs/autointent_classic-light_full_seed{S}/ (и аналоги для pilot / autoembedder)\\n    - Метрики: results/metrics.json (числовые поля + блок extra: embedder, preset, model_dir и т.д.)\\n    - На каждый прогон оценки: runs/metrics_<model_name>_<mode>_seed<S>.json — тот же JSON-объект, что одна строка в metrics.json\\n    - Дублирование метрик в stdout: --print-metrics-json (Kaggle/логи, если файлы не сохранились)\\n\\n    Пример: few-shot без фиксации эмбеддера на всех сидах:\\n        python scripts/run_autointent.py --mode fewshot --n_shots 10 --no-fix-embedder --all-seeds\\n\\nСтабильность на macOS ARM:\\n    load_dotenv и лимиты потоков BLAS/OpenMP должны применяться до import numpy/torch.\\n    Скопируйте .env.example в .env при segfault/sklearn warnings во время AutoML.\\n\\nПрогресс AutoML (по умолчанию):\\n    полоса Optuna (tqdm) по числу trials в текущем узле (scoring/decision);\\n    строка «Overall HPO (estimate)» — грубая доля по этапам HPO (не покрывает долгий первый embed внутри trial).\\n    Отключить: --no-automl-progress\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport logging\\nimport os\\nimport sys\\nfrom collections.abc import Callable\\nfrom dataclasses import dataclass\\nfrom datetime import datetime, timezone\\nfrom pathlib import Path\\nfrom typing import Any\\n\\n# Paths before dotenv / numpy (thread env must be set before BLAS loads).\\nscript_dir = Path(__file__).resolve().parent\\ntask_dir = script_dir.parent\\nproject_root = task_dir.parent.parent\\n\\nfrom dotenv import load_dotenv\\n\\nload_dotenv(project_root / \\".env\\")\\n\\n\\ndef _apply_thread_env_defaults() -> None:\\n    \\"\\"\\"Single-thread BLAS/OpenMP to reduce crashes on macOS (especially Apple Silicon).\\"\\"\\"\\n    defaults = {\\n        \\"OMP_NUM_THREADS\\": \\"1\\",\\n        \\"OPENBLAS_NUM_THREADS\\": \\"1\\",\\n        \\"MKL_NUM_THREADS\\": \\"1\\",\\n        \\"VECLIB_MAXIMUM_THREADS\\": \\"1\\",\\n        \\"NUMEXPR_NUM_THREADS\\": \\"1\\",\\n        \\"LOKY_MAX_CPU_COUNT\\": \\"1\\",\\n        \\"TOKENIZERS_PARALLELISM\\": \\"false\\",\\n    }\\n    for key, value in defaults.items():\\n        if key not in os.environ or os.environ[key] == \\"\\":\\n            os.environ[key] = value\\n\\n\\n_apply_thread_env_defaults()\\n\\nimport numpy as np\\n\\nsys.path.insert(0, str(project_root))\\n\\nfrom autointent import Pipeline, Dataset as AIDataset\\nfrom autointent.configs import LoggingConfig, EmbedderConfig, DataConfig\\n\\nfrom tasks.jailbreak_detection.src.metrics import evaluate_jailbreak\\n\\nlogger = logging.getLogger(__name__)\\n\\n\\ndef _configure_run_logging() -> None:\\n    \\"\\"\\"Один раз настраивает stderr-logging для сообщений об эмбеддере и др.\\"\\"\\"\\n    root = logging.getLogger()\\n    if root.handlers:\\n        return\\n    logging.basicConfig(\\n        level=logging.INFO,\\n        format=\\"%(levelname)s [%(name)s] %(message)s\\",\\n        stream=sys.stderr,\\n    )\\n\\n\\n# Совпадает с prepare_data.DEFAULT_SEEDS и именами wildjailbreak_full100k_seed{S}.json\\nDEFAULT_SEEDS: tuple[int, ...] = (42, 123, 456)\\n\\n\\n@dataclass\\nclass _HpoProgressState:\\n    \\"\\"\\"Tracks Optuna node stages for coarse overall % (classic-light: scoring + decision).\\"\\"\\"\\n\\n    node_total: int = 1\\n    node_index: int = 0\\n\\n\\ndef _make_overall_hpo_callback(n_trials: int | None, state: _HpoProgressState):\\n    \\"\\"\\"Optuna callback: print estimated overall HPO % across NodeOptimizer stages.\\"\\"\\"\\n\\n    def overall_callback(study, trial) -> None:\\n        t = trial.number + 1\\n        denom = n_trials if n_trials is not None else max(t, 1)\\n        span = 100.0 / max(state.node_total, 1)\\n        base = (state.node_index / max(state.node_total, 1)) * 100.0\\n        frac = min(t / float(max(denom, 1)), 1.0)\\n        overall = base + frac * span\\n        print(\\n            f\\"\\\\r[AutoIntent] Overall HPO (estimate): {overall:5.1f}%  —  \\"\\n            f\\"trial {t}/{denom} in current stage    \\",\\n            end=\\"\\",\\n            flush=True,\\n        )\\n\\n    return overall_callback\\n\\n\\ndef _install_automl_progress_hooks(\\n    pipeline: Pipeline,\\n    *,\\n    enable_optuna_bar: bool,\\n) -> Callable[[], None]:\\n    \\"\\"\\"\\n    Optuna hides progress by default; AutoIntent sets WARNING verbosity.\\n    We enable tqdm per study.optimize and print coarse overall % across HPO stages.\\n    \\"\\"\\"\\n    from autointent.nodes._node_optimizer import NodeOptimizer\\n\\n    import autointent.nodes._node_optimizer as _no_mod\\n    import optuna\\n\\n    state = _HpoProgressState()\\n    state.node_total = max(\\n        1,\\n        sum(1 for n in pipeline.nodes.values() if isinstance(n, NodeOptimizer)),\\n    )\\n\\n    Study = optuna.study.Study\\n    _orig_optimize = Study.optimize\\n\\n    def _wrapped_optimize(self, func, *args, **kwargs):\\n        if enable_optuna_bar:\\n            kwargs.setdefault(\\"show_progress_bar\\", True)\\n        n_trials_kw = kwargs.get(\\"n_trials\\")\\n        callbacks = list(kwargs.get(\\"callbacks\\") or [])\\n        callbacks.append(_make_overall_hpo_callback(n_trials_kw, state))\\n        kwargs[\\"callbacks\\"] = callbacks\\n        return _orig_optimize(self, func, *args, **kwargs)\\n\\n    _orig_node_fit = NodeOptimizer.fit\\n\\n    def _wrapped_node_fit(self, context):\\n        print()\\n        stage = state.node_index + 1\\n        label = self.node_info.node_type.value\\n        nt = context.hpo_config.n_trials\\n        print(\\n            f\\"[AutoIntent] HPO stage {stage}/{state.node_total}: {label!r} \\"\\n            f\\"(up to {nt} Optuna trials per stage; bar = trials in this stage)\\",\\n            flush=True,\\n        )\\n        try:\\n            return _orig_node_fit(self, context)\\n        finally:\\n            state.node_index += 1\\n\\n    Study.optimize = _wrapped_optimize  # type: ignore[method-assign]\\n    _no_mod.NodeOptimizer.fit = _wrapped_node_fit  # type: ignore[method-assign]\\n\\n    def _restore() -> None:\\n        Study.optimize = _orig_optimize  # type: ignore[method-assign]\\n        _no_mod.NodeOptimizer.fit = _orig_node_fit  # type: ignore[method-assign]\\n\\n    return _restore\\n\\n\\ndef full_train_filename(seed: int) -> str:\\n    return f\\"wildjailbreak_full100k_seed{seed}.json\\"\\n\\n\\ndef get_data_dir() -> Path:\\n    return task_dir / \\"data\\" / \\"processed\\"\\n\\n\\ndef get_runs_dir() -> Path:\\n    runs_dir = task_dir / \\"runs\\"\\n    runs_dir.mkdir(parents=True, exist_ok=True)\\n    return runs_dir\\n\\n\\ndef get_results_dir() -> Path:\\n    results_dir = task_dir / \\"results\\"\\n    results_dir.mkdir(parents=True, exist_ok=True)\\n    return results_dir\\n\\n\\ndef embedder_hf_model_from_dump(model_dir: Path) -> str | None:\\n    \\"\\"\\"HF id эмбеддера после pipeline.dump (актуально для --no-fix-embedder).\\"\\"\\"\\n    p = model_dir / \\"scoring_module\\" / \\"pydantic\\" / \\"embedder_config\\" / \\"model_dump.json\\"\\n    if not p.is_file():\\n        return None\\n    try:\\n        data = json.loads(p.read_text(encoding=\\"utf-8\\"))\\n    except (json.JSONDecodeError, OSError):\\n        return None\\n    name = data.get(\\"model_name\\")\\n    return name if isinstance(name, str) and name.strip() else None\\n\\n\\ndef decision_module_attrs_from_dump(model_dir: Path) -> dict[str, Any] | None:\\n    \\"\\"\\"Атрибуты decision-модуля из dump (порог и др.) для логов metrics.json.\\"\\"\\"\\n    p = model_dir / \\"decision_module\\" / \\"simple_attrs.json\\"\\n    if not p.is_file():\\n        return None\\n    try:\\n        data = json.loads(p.read_text(encoding=\\"utf-8\\"))\\n    except (json.JSONDecodeError, OSError):\\n        return None\\n    return data if isinstance(data, dict) else None\\n\\n\\ndef confusion_and_rates_jailbreak_positive(\\n    y_true: np.ndarray,\\n    y_pred: np.ndarray,\\n    *,\\n    positive_label: int = 1,\\n) -> dict[str, Any]:\\n    \\"\\"\\"TP/FP/FN/TN и явные FNR/FPR для jailbreak=positive (для cost / HYP-JB-004).\\"\\"\\"\\n    yt = np.asarray(y_true)\\n    yp = np.asarray(y_pred)\\n    pos = positive_label\\n    tp = int(np.sum((yt == pos) & (yp == pos)))\\n    fp = int(np.sum((yt != pos) & (yp == pos)))\\n    fn = int(np.sum((yt == pos) & (yp != pos)))\\n    tn = int(np.sum((yt != pos) & (yp != pos)))\\n    denom_p = tp + fn\\n    denom_n = fp + tn\\n    return {\\n        \\"tp\\": tp,\\n        \\"fp\\": fp,\\n        \\"fn\\": fn,\\n        \\"tn\\": tn,\\n        \\"fnr_jailbreak\\": float(fn / denom_p) if denom_p > 0 else None,\\n        \\"fpr_safe\\": float(fp / denom_n) if denom_n > 0 else None,\\n        \\"n_eval\\": int(len(yt)),\\n        \\"n_safe_true\\": int(np.sum(yt != pos)),\\n        \\"n_jailbreak_true\\": int(np.sum(yt == pos)),\\n    }\\n\\n\\ndef scoring_eval_summary_from_pipeline(\\n    pipeline: Pipeline,\\n    texts: list[str],\\n) -> dict[str, Any] | None:\\n    \\"\\"\\"\\n    Краткая статистика по скорам scoring (margin между классами 0 и 1).\\n    Предполагается порядок классов как в train: 0=safe, 1=jailbreak.\\n    \\"\\"\\"\\n    try:\\n        inf_out = pipeline.predict_with_metadata(texts)\\n        utterances = getattr(inf_out, \\"utterances\\", None)\\n        if not isinstance(utterances, list) or not utterances:\\n            return None\\n        rows = []\\n        for u in utterances:\\n            sc = getattr(u, \\"score\\", None)\\n            if sc is None:\\n                return None\\n            rows.append(np.asarray(sc, dtype=np.float64).ravel())\\n        scores = np.stack(rows, axis=0)\\n    except Exception as exc:\\n        logger.debug(\\"scoring_eval_summary_from_pipeline: %s\\", exc)\\n        return None\\n    if scores.ndim != 2 or scores.shape[1] < 2:\\n        return {\\n            \\"note\\": \\"unexpected_score_shape\\",\\n            \\"shape\\": list(scores.shape),\\n        }\\n    margin = scores[:, 1] - scores[:, 0]\\n    return {\\n        \\"n_scored\\": int(scores.shape[0]),\\n        \\"n_score_dims\\": int(scores.shape[1]),\\n        \\"margin_mean\\": float(np.mean(margin)),\\n        \\"margin_std\\": float(np.std(margin)),\\n        \\"margin_min\\": float(np.min(margin)),\\n        \\"margin_max\\": float(np.max(margin)),\\n        \\"score_col0_mean\\": float(np.mean(scores[:, 0])),\\n        \\"score_col1_mean\\": float(np.mean(scores[:, 1])),\\n        \\"note\\": \\"col0≈safe intent col1≈jailbreak (classic-light binary setup)\\",\\n    }\\n\\n\\ndef get_model_name(pilot: bool, no_fix_embedder: bool) -> str:\\n    if no_fix_embedder:\\n        return \\"autointent_classic-light_autoembedder\\"\\n    if pilot:\\n        return \\"autointent_classic-light_pilot\\"\\n    return \\"autointent_classic-light\\"\\n\\n\\ndef get_embedder_name(pilot: bool) -> str:\\n    if pilot:\\n        return \\"intfloat/multilingual-e5-small\\"\\n    return \\"intfloat/multilingual-e5-large-instruct\\"\\n\\n\\ndef get_model_dir(\\n    pilot: bool,\\n    no_fix_embedder: bool,\\n    mode: str,\\n    n_shots: int | None,\\n    seed: int,\\n) -> Path:\\n    model_name = get_model_name(pilot, no_fix_embedder)\\n    if mode == \\"full\\":\\n        return get_runs_dir() / f\\"{model_name}_full_seed{seed}\\"\\n    assert n_shots is not None\\n    return get_runs_dir() / f\\"{model_name}_{n_shots}shot_seed{seed}\\"\\n\\n\\ndef load_fewshot_train(n_shots: int, seed: int, data_dir: Path) -> dict:\\n    \\"\\"\\"\\n    Load few-shot train data in AutoIntent format.\\n    Format: {\\"intents\\": [{\\"id\\": 0, \\"name\\": \\"safe\\", \\"utterances\\": [...]}],\\n             \\"oos_utterances\\": [...]}\\n    \\"\\"\\"\\n    path = data_dir / f\\"train_shot{n_shots}_seed{seed}.json\\"\\n    with open(path, \\"r\\", encoding=\\"utf-8\\") as f:\\n        return json.load(f)\\n\\n\\ndef load_full_train(seed: int, data_dir: Path) -> dict:\\n    \\"\\"\\"Full-train subset in same AutoIntent dict format as few-shot (prepare_data --full_subset).\\"\\"\\"\\n    path = data_dir / full_train_filename(seed)\\n    if not path.is_file():\\n        raise FileNotFoundError(\\n            f\\"Full-train file missing: {path}\\\\n\\"\\n            \\"Create 100K stratified splits with:\\\\n\\"\\n            \\"  python tasks/jailbreak_detection/scripts/prepare_data.py --full_subset\\\\n\\"\\n            \\"(HF gated dataset + token, or place raw jsonl under data/raw/; see prepare_data.py.)\\"\\n        )\\n    with open(path, \\"r\\", encoding=\\"utf-8\\") as f:\\n        return json.load(f)\\n\\n\\ndef load_test(data_dir: Path) -> dict:\\n    \\"\\"\\"\\n    Load test data.\\n    Format: {\\"utterances\\": [...], \\"labels\\": [...]}\\n    Labels: 0 = safe, 1 = jailbreak\\n    \\"\\"\\"\\n    path = data_dir / \\"test.json\\"\\n    with open(path, \\"r\\", encoding=\\"utf-8\\") as f:\\n        return json.load(f)\\n\\n\\ndef load_eval_binary(data_dir: Path) -> list[dict]:\\n    \\"\\"\\"\\n    Load eval binary data for metrics computation.\\n    Contains: prompt, binary_label, data_type\\n    \\"\\"\\"\\n    path = data_dir / \\"wildjailbreak_eval_binary.jsonl\\"\\n    data = []\\n    with open(path, \\"r\\", encoding=\\"utf-8\\") as f:\\n        for line in f:\\n            data.append(json.loads(line))\\n    return data\\n\\n\\ndef convert_to_autointent_test(test_data: dict) -> list[dict]:\\n    \\"\\"\\"\\n    Convert test data to AutoIntent format for inference.\\n    Binary classification (no OOS):\\n    - safe (label=0): {\\"utterance\\": str, \\"label\\": 0}\\n    - jailbreak (label=1): {\\"utterance\\": str, \\"label\\": 1}\\n    \\"\\"\\"\\n    result = []\\n    for utt, label in zip(test_data[\\"utterances\\"], test_data[\\"labels\\"]):\\n        result.append({\\"utterance\\": utt, \\"label\\": label})\\n    return result\\n\\n\\ndef convert_to_autointent_train(train_data: dict) -> list[dict]:\\n    \\"\\"\\"\\n    Convert few-shot train data to AutoIntent list format.\\n    Binary classification:\\n    - safe (from intents[0]) -> label 0\\n    - jailbreak (from oos_utterances) -> label 1\\n    \\"\\"\\"\\n    result = []\\n    # Safe examples (label=0)\\n    for intent in train_data[\\"intents\\"]:\\n        for utt in intent[\\"utterances\\"]:\\n            result.append({\\"utterance\\": utt, \\"label\\": 0})\\n    # Jailbreak examples (label=1) - NOT OOS, but second intent\\n    for utt in train_data[\\"oos_utterances\\"]:\\n        result.append({\\"utterance\\": utt, \\"label\\": 1})\\n    return result\\n\\n\\ndef save_metrics(result: dict, results_dir: Path) -> None:\\n    \\"\\"\\"Save metrics to metrics.json, appending to existing list.\\"\\"\\"\\n    metrics_path = results_dir / \\"metrics.json\\"\\n\\n    if metrics_path.exists():\\n        with open(metrics_path, \\"r\\", encoding=\\"utf-8\\") as f:\\n            all_results = json.load(f)\\n        if not isinstance(all_results, list):\\n            # Convert dict format to list if needed\\n            all_results = list(all_results.values()) if all_results else []\\n    else:\\n        all_results = []\\n\\n    # Remove existing entry with same model_name, mode, n_shots, seed\\n    all_results = [\\n        r for r in all_results\\n        if not (\\n            r.get(\\"model_name\\") == result[\\"model_name\\"] and\\n            r.get(\\"mode\\") == result[\\"mode\\"] and\\n            r.get(\\"n_shots\\") == result.get(\\"n_shots\\") and\\n            r.get(\\"seed\\") == result.get(\\"seed\\")\\n        )\\n    ]\\n\\n    all_results.append(result)\\n\\n    with open(metrics_path, \\"w\\", encoding=\\"utf-8\\") as f:\\n        json.dump(all_results, f, indent=2, ensure_ascii=False)\\n\\n    print(f\\"Metrics saved to {metrics_path}\\")\\n\\n\\ndef run_metrics_filename(result: dict) -> str:\\n    \\"\\"\\"Stable filename for per-run metrics JSON under runs/ (one object = one metrics.json row).\\"\\"\\"\\n    mn = result[\\"model_name\\"].replace(\\"/\\", \\"_\\").replace(\\" \\", \\"_\\")\\n    mode = result[\\"mode\\"]\\n    seed = result[\\"seed\\"]\\n    if mode == \\"full\\":\\n        return f\\"metrics_{mn}_full_seed{seed}.json\\"\\n    return f\\"metrics_{mn}_{mode}_seed{seed}.json\\"\\n\\n\\ndef save_run_metrics_file(result: dict, runs_dir: Path) -> Path:\\n    \\"\\"\\"Write a single run result under runs/; schema matches one element appended to metrics.json.\\"\\"\\"\\n    runs_dir.mkdir(parents=True, exist_ok=True)\\n    path = runs_dir / run_metrics_filename(result)\\n    with open(path, \\"w\\", encoding=\\"utf-8\\") as f:\\n        json.dump(result, f, indent=2, ensure_ascii=False)\\n    print(f\\"Run metrics saved to {path}\\")\\n    return path\\n\\n\\ndef train(args, data_dir: Path, model_dir: Path) -> None:\\n    \\"\\"\\"Train AutoIntent pipeline.\\"\\"\\"\\n    no_fix_embedder = getattr(args, \\"no_fix_embedder\\", False)\\n    mode = getattr(args, \\"mode\\", \\"fewshot\\")\\n    if no_fix_embedder:\\n        embedder_name = \\"auto (optimized by AutoML)\\"\\n    else:\\n        embedder_name = get_embedder_name(args.pilot)\\n\\n    print(\\"=\\" * 60)\\n    print(\\"AutoIntent Training (Jailbreak Detection)\\")\\n    print(\\"=\\" * 60)\\n    if no_fix_embedder:\\n        mode_label = \\"AUTO-EMBEDDER\\"\\n    else:\\n        mode_label = \\"PILOT\\" if args.pilot else \\"FINAL\\"\\n    print(f\\"Mode: {mode_label}\\")\\n    print(f\\"Embedder: {embedder_name}\\")\\n    if no_fix_embedder:\\n        logger.info(\\n            \\"Embedder policy: AutoML (preset classic-light); \\"\\n            \\"конкретный HuggingFace id будет известен после dump в конце обучения.\\",\\n        )\\n    else:\\n        logger.info(\\n            \\"Embedder policy: фиксированный; HuggingFace model_name=%s\\",\\n            embedder_name,\\n        )\\n    if mode == \\"full\\":\\n        print(f\\"Training: FULL ({full_train_filename(args.seed)}), seed={args.seed}\\")\\n    else:\\n        print(f\\"Training: {args.n_shots}shot, seed={args.seed}\\")\\n    print(f\\"Output: {model_dir}\\")\\n    print(\\"=\\" * 60)\\n    print()\\n\\n    # Load data\\n    if mode == \\"full\\":\\n        train_raw = load_full_train(args.seed, data_dir)\\n    else:\\n        train_raw = load_fewshot_train(args.n_shots, args.seed, data_dir)\\n    test_raw = load_test(data_dir)\\n\\n    train_ai = convert_to_autointent_train(train_raw)\\n    test_ai = convert_to_autointent_test(test_raw)\\n\\n    # Binary classification: 2 intents (no OOS)\\n    intents = [\\n        {\\"id\\": 0, \\"name\\": \\"safe\\"},\\n        {\\"id\\": 1, \\"name\\": \\"jailbreak\\"},\\n    ]\\n\\n    n_safe = len(train_raw[\\"intents\\"][0][\\"utterances\\"])\\n    n_jailbreak = len(train_raw[\\"oos_utterances\\"])\\n    print(f\\"Train: {len(train_ai)} samples\\")\\n    print(f\\"  - safe (label=0): {n_safe}\\")\\n    print(f\\"  - jailbreak (label=1): {n_jailbreak}\\")\\n    print(f\\"Test: {len(test_ai)} samples\\")\\n    print()\\n\\n    # Create AutoIntent Dataset\\n    ai_dataset = AIDataset.from_dict({\\n        \\"train\\": train_ai,\\n        \\"test\\": test_ai,\\n        \\"intents\\": intents,\\n    })\\n\\n    # Create pipeline with classic-light preset\\n    pipeline = Pipeline.from_preset(\\"classic-light\\")\\n    if not no_fix_embedder:\\n        pipeline.set_config(EmbedderConfig(model_name=embedder_name))\\n\\n    # Cross-validation only for few-shot (full train uses default scheme)\\n    if mode == \\"fewshot\\":\\n        pipeline.set_config(DataConfig(scheme=\\"cv\\", n_folds=3))\\n\\n    # Logging config\\n    pipeline.set_config(LoggingConfig(\\n        project_dir=model_dir,\\n        dump_modules=True,\\n        clear_ram=False,\\n    ))\\n\\n    # Train\\n    print(\\"Starting AutoML optimization...\\")\\n    print(\\"This may take a while...\\")\\n    print()\\n\\n    show_progress = not getattr(args, \\"no_automl_progress\\", False)\\n    restore_hooks: Callable[[], None] | None = None\\n    try:\\n        if show_progress:\\n            restore_hooks = _install_automl_progress_hooks(\\n                pipeline,\\n                enable_optuna_bar=True,\\n            )\\n        pipeline.fit(ai_dataset)\\n    finally:\\n        if restore_hooks is not None:\\n            print()\\n            restore_hooks()\\n\\n    print()\\n    print(\\"AutoML optimization completed!\\")\\n\\n    # Save model\\n    print(f\\"Saving model to {model_dir}...\\")\\n    pipeline.dump(model_dir)\\n\\n    # Save metadata\\n    if mode == \\"full\\":\\n        meta_mode = \\"full\\"\\n        meta_n_shots = None\\n    else:\\n        meta_mode = f\\"{args.n_shots}shot\\"\\n        meta_n_shots = args.n_shots\\n\\n    metadata = {\\n        \\"model_name\\": get_model_name(args.pilot, no_fix_embedder),\\n        \\"mode\\": meta_mode,\\n        \\"n_shots\\": meta_n_shots,\\n        \\"seed\\": args.seed,\\n        \\"embedder\\": embedder_name,\\n        \\"embedder_fixed\\": not no_fix_embedder,\\n        \\"pilot\\": args.pilot,\\n        \\"preset\\": \\"classic-light\\",\\n        \\"task\\": \\"jailbreak_detection\\",\\n        \\"approach\\": \\"binary_classification\\",  # Not OOS detection\\n    }\\n    resolved_hf = embedder_hf_model_from_dump(model_dir)\\n    if resolved_hf:\\n        metadata[\\"embedder_hf_model\\"] = resolved_hf\\n        print(f\\"Chosen HuggingFace embedder (from dump): {resolved_hf}\\")\\n        logger.info(\\n            \\"В сохранённом пайплайне эмбеддер (HF model_name): %s\\",\\n            resolved_hf,\\n        )\\n    else:\\n        logger.warning(\\n            \\"Не удалось прочитать HF model_name из \\"\\n            \\"scoring_module/pydantic/embedder_config/model_dump.json под %s\\",\\n            model_dir,\\n        )\\n\\n    (model_dir / \\"train_metadata.json\\").write_text(\\n        json.dumps(metadata, indent=2, ensure_ascii=False),\\n        encoding=\\"utf-8\\",\\n    )\\n\\n    print()\\n    print(\\"Training completed!\\")\\n\\n\\ndef evaluate(\\n    args,\\n    data_dir: Path,\\n    model_dir: Path,\\n    results_dir: Path,\\n    runs_dir: Path | None = None,\\n) -> None:\\n    \\"\\"\\"Evaluate trained AutoIntent pipeline.\\"\\"\\"\\n    print()\\n    print(\\"=\\" * 60)\\n    print(\\"AutoIntent Evaluation (Jailbreak Detection)\\")\\n    print(\\"=\\" * 60)\\n\\n    # Load metadata\\n    metadata_path = model_dir / \\"train_metadata.json\\"\\n    no_fix_embedder = getattr(args, \\"no_fix_embedder\\", False)\\n    mode = getattr(args, \\"mode\\", \\"fewshot\\")\\n    if metadata_path.exists():\\n        metadata = json.loads(metadata_path.read_text())\\n    else:\\n        if mode == \\"full\\":\\n            meta_mode = \\"full\\"\\n            meta_n_shots = None\\n        else:\\n            meta_mode = f\\"{args.n_shots}shot\\"\\n            meta_n_shots = args.n_shots\\n        metadata = {\\n            \\"model_name\\": get_model_name(args.pilot, no_fix_embedder),\\n            \\"mode\\": meta_mode,\\n            \\"n_shots\\": meta_n_shots,\\n            \\"seed\\": args.seed,\\n            \\"embedder\\": (\\n                \\"auto (optimized by AutoML)\\"\\n                if no_fix_embedder\\n                else get_embedder_name(args.pilot)\\n            ),\\n            \\"embedder_fixed\\": not no_fix_embedder,\\n            \\"pilot\\": args.pilot,\\n        }\\n\\n    hf_emb = metadata.get(\\"embedder_hf_model\\") or embedder_hf_model_from_dump(model_dir)\\n\\n    print(f\\"Model: {metadata[\'model_name\']}\\")\\n    print(f\\"Mode: {metadata[\'mode\']}\\")\\n    print(f\\"Embedder: {metadata[\'embedder\']}\\")\\n    if hf_emb:\\n        print(f\\"HuggingFace embedder model: {hf_emb}\\")\\n        logger.info(\\"Оценка: используется HF эмбеддер model_name=%s\\", hf_emb)\\n    elif metadata.get(\\"embedder_fixed\\", True):\\n        logger.info(\\n            \\"Оценка: по метаданным фиксированный режим; HF model из metadata/dump не повторён.\\",\\n        )\\n    else:\\n        logger.warning(\\n            \\"Оценка: autoembedder, но HF model_name не найден в metadata и dump (%s)\\",\\n            model_dir,\\n        )\\n    print(\\"=\\" * 60)\\n    print()\\n\\n    # Load model\\n    print(f\\"Loading model from {model_dir}...\\")\\n    pipeline = Pipeline.load(model_dir)\\n    print(\\"Model loaded!\\")\\n    print()\\n\\n    # Load test data\\n    test_raw = load_test(data_dir)\\n    test_texts = test_raw[\\"utterances\\"]\\n\\n    # Load eval binary for data_types and ground truth\\n    eval_binary = load_eval_binary(data_dir)\\n    y_true = np.array([1 if r[\\"binary_label\\"] == \\"jailbreak\\" else 0 for r in eval_binary])\\n    data_types = np.array([r[\\"data_type\\"] for r in eval_binary])\\n\\n    print(f\\"Test samples: {len(test_texts)}\\")\\n    print(f\\"  safe: {sum(y_true == 0)}, jailbreak: {sum(y_true == 1)}\\")\\n    print()\\n\\n    # Predictions\\n    print(\\"Running predictions...\\")\\n    raw_preds = pipeline.predict(test_texts)\\n\\n    # Binary classification: predictions are 0 (safe) or 1 (jailbreak) directly\\n    # None means model couldn\'t decide - treat as jailbreak (safer for guardrail)\\n    y_pred = np.array([1 if p is None else p for p in raw_preds])\\n\\n    print(f\\"Predictions: {len(y_pred)}\\")\\n    print(f\\"  predicted safe: {sum(y_pred == 0)}\\")\\n    print(f\\"  predicted jailbreak: {sum(y_pred == 1)}\\")\\n    print()\\n\\n    # Compute metrics\\n    print(\\"Computing metrics...\\")\\n    metrics = evaluate_jailbreak(y_true, y_pred, data_types, oos_label=1)\\n\\n    eval_counts = confusion_and_rates_jailbreak_positive(y_true, y_pred, positive_label=1)\\n    decision_attrs_raw = decision_module_attrs_from_dump(model_dir)\\n    decision_attrs_json: dict[str, Any] | None\\n    if decision_attrs_raw is None:\\n        decision_attrs_json = None\\n    else:\\n        try:\\n            dumped = json.dumps(decision_attrs_raw)\\n            if len(dumped) > 12_000:\\n                decision_attrs_json = {\\n                    \\"_truncated\\": True,\\n                    \\"top_level_keys\\": list(decision_attrs_raw.keys()),\\n                }\\n            else:\\n                decision_attrs_json = decision_attrs_raw\\n        except (TypeError, ValueError):\\n            decision_attrs_json = {\\"_error\\": \\"non_serializable_decision_attrs\\"}\\n\\n    scores_eval_summary = scoring_eval_summary_from_pipeline(pipeline, test_texts)\\n\\n    # Print results\\n    print()\\n    print(\\"=\\" * 60)\\n    print(f\\"Results ({metadata[\'mode\']})\\")\\n    print(\\"=\\" * 60)\\n    for k, v in metrics.items():\\n        if v is not None:\\n            print(f\\"  {k}: {v:.4f}\\")\\n\\n    # Save results (extra — воспроизводимость + данные для HYP-JB-004 / порог / скоры)\\n    result = {\\n        \\"model_name\\": metadata[\\"model_name\\"],\\n        \\"mode\\": metadata[\\"mode\\"],\\n        \\"n_shots\\": metadata[\\"n_shots\\"],\\n        \\"seed\\": metadata[\\"seed\\"],\\n        \\"f1\\": metrics[\\"f1\\"],\\n        \\"precision\\": metrics[\\"precision\\"],\\n        \\"recall\\": metrics[\\"recall\\"],\\n        \\"over_refusal_rate\\": metrics[\\"over_refusal_rate\\"],\\n        \\"recall_adversarial_harmful\\": metrics.get(\\"recall_adversarial_harmful\\"),\\n        \\"timestamp\\": datetime.now(timezone.utc).isoformat(),\\n        \\"extra\\": {\\n            \\"preset\\": metadata.get(\\"preset\\", \\"classic-light\\"),\\n            \\"embedder\\": metadata[\\"embedder\\"],\\n            \\"embedder_hf_model\\": hf_emb,\\n            \\"embedder_fixed\\": metadata.get(\\"embedder_fixed\\", True),\\n            \\"pilot\\": metadata[\\"pilot\\"],\\n            \\"model_dir\\": str(model_dir),\\n            \\"eval_counts\\": eval_counts,\\n            \\"decision_module_attrs\\": decision_attrs_json,\\n            \\"scores_eval_summary\\": scores_eval_summary,\\n        },\\n    }\\n\\n    save_metrics(result, results_dir)\\n    save_run_metrics_file(result, runs_dir if runs_dir is not None else get_runs_dir())\\n\\n    if getattr(args, \\"print_hypothesis_log\\", False):\\n        hyp_payload = {\\n            \\"model_name\\": result[\\"model_name\\"],\\n            \\"mode\\": result[\\"mode\\"],\\n            \\"seed\\": result[\\"seed\\"],\\n            \\"n_shots\\": result[\\"n_shots\\"],\\n            \\"metrics_core\\": {\\n                \\"f1\\": result[\\"f1\\"],\\n                \\"recall\\": result[\\"recall\\"],\\n                \\"over_refusal_rate\\": result[\\"over_refusal_rate\\"],\\n                \\"precision\\": result[\\"precision\\"],\\n            },\\n            \\"embedder_hf_model\\": hf_emb,\\n            \\"eval_counts\\": eval_counts,\\n            \\"scores_eval_summary\\": scores_eval_summary,\\n            \\"decision_module_attrs\\": decision_attrs_json,\\n            \\"hypothesis_notes\\": (\\n                \\"FNR≈fnr_jailbreak, FPR_safe≈fpr_safe; для асимметричного порога нужен \\"\\n                \\"sweep по threshold на val (не только эта точка).\\"\\n            ),\\n        }\\n        print()\\n        print(\\"=\\" * 60)\\n        print(\\"HYPOTHESIS_LOG (краткий JSON для отчёта / Kaggle output)\\")\\n        print(\\"=\\" * 60)\\n        print(json.dumps(hyp_payload, indent=2, ensure_ascii=False))\\n\\n    if getattr(args, \\"print_metrics_json\\", False):\\n        print()\\n        print(\\"=\\" * 60)\\n        print(\\"METRICS_JSON (same object as metrics.json row)\\")\\n        print(\\"=\\" * 60)\\n        print(json.dumps(result, indent=2, ensure_ascii=False))\\n\\n    if metadata.get(\\"pilot\\") and metadata.get(\\"embedder_fixed\\", True):\\n        print()\\n        print(\\"=\\" * 60)\\n        print(\\"PILOT run: using small embedder for fast validation\\")\\n        print(\\"Re-run without --pilot for final results\\")\\n        print(\\"=\\" * 60)\\n\\n\\ndef main():\\n    _configure_run_logging()\\n\\n    parser = argparse.ArgumentParser(\\n        description=\\"Run AutoIntent on WildJailbreak (train + evaluate)\\"\\n    )\\n    parser.add_argument(\\n        \\"--mode\\",\\n        choices=[\\"fewshot\\", \\"full\\"],\\n        default=\\"fewshot\\",\\n        help=(\\n            \\"fewshot: train_shot{N}_seed{S}.json; \\"\\n            \\"full: wildjailbreak_full100k_seed{S}.json (prepare_data --full_subset)\\"\\n        ),\\n    )\\n    parser.add_argument(\\n        \\"--n_shots\\",\\n        type=int,\\n        choices=[10, 20, 50],\\n        default=10,\\n        help=\\"Number of shots per class\\",\\n    )\\n    parser.add_argument(\\n        \\"--seed\\",\\n        type=int,\\n        choices=[42, 123, 456],\\n        default=42,\\n        help=\\"Random seed\\",\\n    )\\n    parser.add_argument(\\n        \\"--pilot\\",\\n        action=\\"store_true\\",\\n        help=\\"Use small embedder for fast validation\\",\\n    )\\n    parser.add_argument(\\n        \\"--no-fix-embedder\\",\\n        action=\\"store_true\\",\\n        help=\\"Let AutoML optimize embedder (slower, potentially better)\\",\\n    )\\n    parser.add_argument(\\n        \\"--train-only\\",\\n        action=\\"store_true\\",\\n        help=\\"Only train, skip evaluation\\",\\n    )\\n    parser.add_argument(\\n        \\"--eval-only\\",\\n        action=\\"store_true\\",\\n        help=\\"Only evaluate (model must exist)\\",\\n    )\\n    parser.add_argument(\\n        \\"--all-seeds\\",\\n        action=\\"store_true\\",\\n        help=f\\"Прогнать подряд все сиды {list(DEFAULT_SEEDS)} (игнорирует одиночный --seed)\\",\\n    )\\n    parser.add_argument(\\n        \\"--no-automl-progress\\",\\n        action=\\"store_true\\",\\n        help=(\\n            \\"Отключить tqdm Optuna и строку Overall HPO %% во время AutoML \\"\\n            \\"(полезно для логов без перезаписи строк)\\"\\n        ),\\n    )\\n    parser.add_argument(\\n        \\"--print-metrics-json\\",\\n        action=\\"store_true\\",\\n        help=\\"После каждой оценки вывести полный JSON строки метрик в stdout (удобно для Kaggle/логов)\\",\\n    )\\n    parser.add_argument(\\n        \\"--print-hypothesis-log\\",\\n        action=\\"store_true\\",\\n        help=(\\n            \\"После оценки вывести компактный JSON (метрики, eval_counts, скоры, decision attrs) \\"\\n            \\"для отчёта по гипотезам (Kaggle output)\\"\\n        ),\\n    )\\n    args = parser.parse_args()\\n\\n    seeds = list(DEFAULT_SEEDS) if args.all_seeds else [args.seed]\\n\\n    data_dir = get_data_dir()\\n    results_dir = get_results_dir()\\n\\n    for run_idx, seed in enumerate(seeds, start=1):\\n        args.seed = seed\\n        model_dir = get_model_dir(\\n            args.pilot,\\n            args.no_fix_embedder,\\n            args.mode,\\n            args.n_shots if args.mode == \\"fewshot\\" else None,\\n            args.seed,\\n        )\\n\\n        if len(seeds) > 1:\\n            print()\\n            print(\\"#\\" * 60)\\n            print(f\\"# Seed {args.seed}  ({run_idx}/{len(seeds)})\\")\\n            print(\\"#\\" * 60)\\n\\n        # Train\\n        if not args.eval_only:\\n            train(args, data_dir, model_dir)\\n\\n        # Evaluate\\n        if not args.train_only:\\n            if not model_dir.exists():\\n                print(f\\"Error: Model directory not found: {model_dir}\\")\\n                print(\\"Train a model first (remove --eval-only flag)\\")\\n                sys.exit(1)\\n            evaluate(args, data_dir, model_dir, results_dir)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n"}')
for rel, txt in _P.items():
    p = PROJECT_ROOT / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(txt, encoding="utf-8")
print("Written", len(_P), "files under", PROJECT_ROOT)



## 4. HF токен и потоки



In [ ]:
# ========== Вставьте Hugging Face read-token сюда ==========
HF_TOKEN = ""


def apply_hf_token() -> None:
    cell_tok = (HF_TOKEN if isinstance(HF_TOKEN, str) else "").strip()
    if cell_tok:
        os.environ["HF_TOKEN"] = cell_tok
        print("HF_TOKEN задан из ячейки (длина", len(cell_tok), "символов).")
        return
    if os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN"):
        print("HF_TOKEN уже задан в окружении.")
        return
    if IS_KAGGLE:
        from kaggle_secrets import UserSecretsClient

        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
        print("HF_TOKEN загружен из Kaggle Secrets.")
    else:
        import getpass

        os.environ["HF_TOKEN"] = getpass.getpass("HF token (read): ").strip()


apply_hf_token()

for k, v in {
    "OMP_NUM_THREADS": "1",
    "OPENBLAS_NUM_THREADS": "1",
    "MKL_NUM_THREADS": "1",
    "VECLIB_MAXIMUM_THREADS": "1",
    "NUMEXPR_NUM_THREADS": "1",
    "LOKY_MAX_CPU_COUNT": "1",
    "TOKENIZERS_PARALLELISM": "false",
}.items():
    os.environ.setdefault(k, v)

print("OMP_NUM_THREADS=", os.environ.get("OMP_NUM_THREADS"))



## 5. Данные: база + full subset



In [ ]:
def run_script(rel_script: str, extra_args: list[str] | None = None) -> None:
    script = PROJECT_ROOT / rel_script
    cmd = [sys.executable, str(script)] + (extra_args or [])
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True, cwd=str(PROJECT_ROOT), env=os.environ.copy())


run_script("tasks/jailbreak_detection/scripts/prepare_data.py")



In [ ]:
run_script(
    "tasks/jailbreak_detection/scripts/prepare_data.py",
    ["--full_subset"],
)



## 6a. AutoIntent **few-shot** (10, 20, 50) × все сиды

`--no-fix-embedder`, `--print-metrics-json`, `--print-hypothesis-log`.



In [ ]:
# Few-shot: отдельно 10 / 20 / 50 shots × все сиды (42, 123, 456).
# В stdout: METRICS_JSON + HYPOTHESIS_LOG (eval_counts, scores_eval_summary, decision attrs).
FEWSHOT_NS = (10, 20, 50)
for n in FEWSHOT_NS:
    print("\n" + "#" * 70)
    print(f"# FEWSHOT n_shots={n} — все сиды")
    print("#" * 70 + "\n")
    run_script(
        "tasks/jailbreak_detection/scripts/run_autointent.py",
        [
            "--mode", "fewshot",
            "--n_shots", str(n),
            "--all-seeds",
            "--no-fix-embedder",
            "--print-metrics-json",
            "--print-hypothesis-log",
        ],
    )



## 6b. AutoIntent **full** × все сиды

Очень долго. Запускайте отдельно при необходимости.



In [ ]:
# Full train: отдельная ячейка (долго). Те же флаги логирования.
print("\n" + "#" * 70)
print("# MODE FULL — все сиды")
print("#" * 70 + "\n")
run_script(
    "tasks/jailbreak_detection/scripts/run_autointent.py",
    [
        "--mode", "full",
        "--all-seeds",
        "--no-fix-embedder",
        "--print-metrics-json",
        "--print-hypothesis-log",
    ],
)



## 7. Хвост `metrics.json`



In [ ]:
import json

metrics_path = TASK_ROOT / "results" / "metrics.json"
if metrics_path.is_file():
    rows = json.loads(metrics_path.read_text(encoding="utf-8"))
    print(f"metrics.json: всего строк={len(rows)}")
    tail = rows[-min(8, len(rows)) :]
    print("\n--- Последние строки (extra.eval_counts, scores_eval_summary, ...) ---\n")
    for r in tail:
        print(json.dumps(r, indent=2, ensure_ascii=False))
        print()
else:
    print("Файл metrics.json не найден:", metrics_path)

